# Training `penrose_diffusion` on Colab

**Before running anything:** Runtime -> Change runtime type -> **T4 GPU**. Cell 4 verifies this; if it reports `IS_GPU: False` you are on CPU and a run will take many hours instead of tens of minutes.

## Cost of a run

Dataset size is `num_classes * samples_per_class * num_copies` (`code/data/create.py:22`). With MPEG7 that is `70 * 20 * num_copies`, so the default `num_copies=100` gives **140,000 samples** -> ~2,187 batches/epoch -> ~221,000 steps at `num_epochs=101`. That is roughly **1-2 hours per run on a T4**, which exceeds what a free Colab session comfortably allows for a multi-run sweep.

For comparing hyperparameters, build the dataset with `num_copies=10` instead (14,000 samples). Every run gets ~10x cheaper and an A/B stays perfectly valid. Cell 5 does this.

## 1. Mount Drive

Not optional. `code/compatibility.py:65-71` calls `sys.exit(1)` on Colab if `/content/drive` is absent. It is also what persists checkpoints to `/content/drive/MyDrive/penrose_diffusion` across the disconnects you should expect on hour-long runs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone

Point `REPO` / `BRANCH` at a fork if you are testing an unmerged branch.

In [ ]:
REPO   = 'https://github.com/rakeshvar/penrose_diffusion'
BRANCH = 'master'

!git clone -b $BRANCH $REPO
%cd penrose_diffusion

## 3. Dependencies

Colab already ships torch (CUDA), numpy, scipy, tqdm and pyyaml, so only `wandb` is missing. Skip it entirely if you intend to train with `-w enable=False`.

In [ ]:
!pip install -q wandb

## 4. Check the accelerator

Worth ten seconds before committing an hour. Want `IS_GPU: True`.

In [ ]:
import code.compatibility as compat
compat.print_env()

## 5. Get a dataset

Two routes. **Route A is strongly preferred** once you have generated a dataset even once.

### Route A - copy a prebuilt `.npz` from Drive

Instant, and avoids needing MPEG7 on the Colab machine at all.

In [ ]:
DATASET = 'hexxy_t096_c10_u18.npz'

!mkdir -p datasets
!cp /content/drive/MyDrive/penrose_diffusion/$DATASET datasets/
!ls -la datasets/

### Route B - generate from MPEG7

`scripts/create_dataset.py:72` hardcodes `folder = "library/MPEG7/gifs"`, and that directory is **not** in the repo. You must supply the MPEG7 shape dataset (70 classes x 20 gifs) at that path yourself; the cell below assumes you keep a copy on Drive.

Output naming, from `code/data/create.py:57` plus the prefix logic at `scripts/create_dataset.py:64-68`: `{hex|pen}{xy}{ind}_t{tiles:03d}_c{copies:02d}_u{100*unit_side:02d}.npz`. Since `save_xyac` defaults to True the prefix is `hexxy`, not `hex` — so `6 96 10 0.18` writes `datasets/hexxy_t096_c10_u18.npz`.

Copy the result back to Drive so you only ever pay this cost once.

In [ ]:
# Requires library/MPEG7/gifs to exist first, e.g.:
# !mkdir -p library/MPEG7 && cp -r /content/drive/MyDrive/MPEG7/gifs library/MPEG7/

# symmetry=6 (hex), num_tiles=96, num_copies=10, unit_side=0.18
!python -m scripts.create_dataset 6 96 10 0.18

!cp datasets/*.npz /content/drive/MyDrive/penrose_diffusion/

## 6. Train

Drop `-w enable=False` and run `wandb.login()` first if you want logging — on runs this long, and given Colab disconnects, it is genuinely worth having.

In [ ]:
!python train.py dd128 datasets/$DATASET -w enable=False

## 7. Optional: stability-margin sweep

Only meaningful on a branch that has `margin_lambda`. Training is unseeded (there is no seed flag anywhere in the repo), so re-running an identical command gives an independent replicate.

Compare arms on **`loss/lattice_sample`**, which is computed from generated samples. Do *not* compare `loss/avg_loss` across arms — the `margin_lambda > 0` runs are optimizing a different objective, so that number is not comparable between them.

In [ ]:
for lam in ['0.0', '0.02', '0.05']:
    print(f'===== margin_lambda={lam} =====')
    !python train.py dd128 datasets/$DATASET -m margin_lambda=$lam -w run_name=mrg_$lam

## 8. Keep the outputs

On Colab `OUTPUT_BASE_DIR` is already `/content/drive/MyDrive/penrose_diffusion` (`code/compatibility.py:72`), so checkpoints and sample SVGs land on Drive automatically. This cell is only needed if you overrode it with `-o`.

In [ ]:
!ls -la /content/drive/MyDrive/penrose_diffusion/